# Libraries

In [ ]:
# Reinstall PyTorch stack with CUDA 12.8 support
!pip uninstall -y torch torchvision torchaudio torchao
!pip install -q \
    torch==2.10.0+cu128 \
    torchvision==0.25.0+cu128 \
    torchaudio==2.10.0+cu128 \
    torchao==0.10.0 \
    --index-url https://download.pytorch.org/whl/cu128

# vLLM
!pip install -q vllm==0.19.0

# Dependencies
!pip install -q "protobuf>=5.26.1,<6.0.0" --break-system-packages

!pip install -q llmcompressor==0.10.0.2
!pip install -q compressed-tensors==0.14.0.1

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 916.9/916.9 MB 1.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 4.4 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 2.2 MB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 12.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Standard library imports
import os
import gc
import json
import time
import random
import shutil
import subprocess

# Third-party libraries
import numpy as np
from vllm import LLM, SamplingParams
from huggingface_hub import login
from transformers import AutoTokenizer, set_seed

# Deep learning framework
import torch

2026-05-22 12:27:46.594301: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779452866.828871      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779452866.896563      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779452867.454852      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779452867.454903      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779452867.454906      57 computation_placer.cc:177] computation placer alr

# Global Variables

In [ ]:
# User & Repository Configuration
USER_NAME = ""
EMAIL     = ""
REPO      = "MahmoudOsama20/EdgeCompress"  # Target repository

# Authentication Token
GIT_TOKEN = ""
HF_TOKEN = ""

# Model Selection
MODEL = "Qwen3-4B"

# Model & Tokenizer Configuration
MODEL_ID     = ""
TOKENIZER_ID = "Qwen/Qwen3-4B"

MODEL_NAME           = "Qwen3-4B"
MODEL_NAME_IN_REPO   = "Qwen3-4B-WANDA-AWQ"
COMPRESSION_METHOD   = "WANDA & AWQ"
MODEL_TYPE           = "Pruning & Quantization"
SPARSITY = 25
#PATH = f"Models/{SPARSITY}"

# Inference Prompt (Chat Format)
PROMPT = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "Provide a concise explanation of Artificial Intelligence."
    }
]

dummy_prompt = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "Give me story of a brave man"
    }
]

# Generation Configuration
MAX_GENERATION_TOKENS = 150
SEED = 42

# Output Configuration
OUTPUT_FILE = "model_metrics.json"

# Functions

In [4]:
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Seeding

In [5]:
set_reproducibility(SEED)

# Huggingface Login

In [ ]:
login(HF_TOKEN)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# Prompt Preprocessing

**DummyPrompt Tokenization**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)

#Chat Formating
dummy_prompt_text = tokenizer.apply_chat_template(
    dummy_prompt, 
    tokenize=False, 
    add_generation_prompt=True,
    enable_thinking=False
)

#Tokenization
dummy_prompt_token_ids = tokenizer.encode(dummy_prompt_text)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

**RealPrompt Tokenization**

In [ ]:
#Chat Formating
prompt_text = tokenizer.apply_chat_template(
    PROMPT, 
    tokenize=False, 
    add_generation_prompt=True,
    enable_thinking=False
)

#Tokenization
prompt_token_ids = tokenizer.encode(prompt_text)

# Initializing vLLM

In [ ]:
llm = LLM(
    model=MODEL_ID,
    tokenizer=TOKENIZER_ID,
    dtype="float16",
    max_model_len=256,
    tensor_parallel_size=1,
    gpu_memory_utilization=0.80,
    attention_backend = "TRITON_ATTN",
    disable_log_stats=False,
    enable_prefix_caching = False
)
print("vLLM Initialized Successfully!")


sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_GENERATION_TOKENS,
    min_tokens=MAX_GENERATION_TOKENS,
    seed = SEED
)

ttft_sampling_params = SamplingParams(
    temperature=0.0,
    seed = SEED,
    max_tokens=1,
    ignore_eos=True # Ensures it doesn't accidentally stop early
)

INFO 05-22 12:28:16 [utils.py:233] non-default args: {'tokenizer': 'microsoft/Phi-4-mini-instruct', 'dtype': 'float16', 'max_model_len': 256, 'tensor_parallel_size': 2, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.85, 'attention_backend': 'TRITON_ATTN', 'model': 'microsoft/Phi-4-mini-instruct'}


config.json: 0.00B [00:00, ?B/s]

INFO 05-22 12:28:16 [config.py:437] Replacing legacy 'type' key with 'rope_type'
INFO 05-22 12:28:38 [model.py:533] Resolved architecture: Phi3ForCausalLM
WARNING 05-22 12:28:38 [model.py:1920] Casting torch.bfloat16 to torch.float16.
INFO 05-22 12:28:38 [model.py:1582] Using max model len 256
INFO 05-22 12:28:39 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-22 12:28:39 [vllm.py:754] Asynchronous scheduling is enabled.


generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

WARNING 05-22 12:28:40 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized


...

(Worker_TP0 pid=259) INFO 05-22 12:30:10 [default_loader.py:384] Loading weights took 13.96 seconds
(Worker_TP0 pid=259) INFO 05-22 12:30:11 [gpu_model_runner.py:4566] Model loading took 3.63 GiB memory and 50.828511 seconds
(Worker_TP0 pid=259) INFO 05-22 12:30:22 [backends.py:988] Using cache directory: /root/.cache/vllm/torch_compile_cache/1e79882f3c/rank_0_0/backbone for vLLM's torch.compile
(Worker_TP0 pid=259) INFO 05-22 12:30:22 [backends.py:1048] Dynamo bytecode transform time: 10.48 s


(Worker_TP0 pid=259) [rank0]:W0522 12:30:24.112000 259 torch/_inductor/utils.py:1679] Not enough SMs to use max_autotune_gemm mode
(Worker_TP1 pid=260) [rank1]:W0522 12:30:24.120000 260 torch/_inductor/utils.py:1679] Not enough SMs to use max_autotune_gemm mode


(Worker_TP0 pid=259) INFO 05-22 12:30:29 [backends.py:371] Cache the graph of compile range (1, 8192) for later use
(Worker_TP1 pid=260) INFO 05-22 12:30:29 [backends.py:371] Cache the graph of compile range (1, 8192) for later use
(Worker_TP0 pid=259) INFO 05-22 12:30:35 [backends.py:387] Compiling a graph for compile range (1, 8192) takes 12.60 s
(Worker_TP0 pid=259) INFO 05-22 12:30:37 [decorators.py:627] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/f87fbe634706e58f23ddaf7a9d073e0ea21c155cfb734ae1e6c2701b243bc455/rank_0_0/model
(Worker_TP0 pid=259) INFO 05-22 12:30:37 [monitor.py:48] torch.compile took 25.20 s in total
(Worker_TP0 pid=259) INFO 05-22 12:30:39 [monitor.py:76] Initial profiling/warmup run took 2.07 s
(Worker_TP1 pid=260) INFO 05-22 12:30:50 [kv_cache_utils.py:826] Overriding num_gpu_blocks=0 with num_gpu_blocks_override=512
(Worker_TP1 pid=260) INFO 05-22 12:30:50 [gpu_model_runner.py:5607] Profiling CUDA graph memory: PIECEWI

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:05<00:00,  8.88it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:07<00:00,  4.48it/s]


(Worker_TP0 pid=259) INFO 05-22 12:31:11 [gpu_model_runner.py:5746] Graph capturing finished in 15 secs, took 1.10 GiB
(Worker_TP0 pid=259) INFO 05-22 12:31:11 [gpu_worker.py:617] CUDA graph pool memory: 1.1 GiB (actual), 1.19 GiB (estimated), difference: 0.09 GiB (8.0%).
(Worker_TP1 pid=260) INFO 05-22 12:31:11 [gpu_worker.py:617] CUDA graph pool memory: 1.1 GiB (actual), 1.19 GiB (estimated), difference: 0.09 GiB (8.0%).
(EngineCore pid=234) INFO 05-22 12:31:11 [core.py:281] init engine (profile, create kv cache, warmup model) took 59.76 seconds
(EngineCore pid=234) INFO 05-22 12:31:14 [vllm.py:754] Asynchronous scheduling is enabled.
INFO 05-22 12:31:14 [llm.py:391] Supported tasks: ['generate']
vLLM Initialized Successfully!


# Inference

In [12]:
# ── Initialize collectors ──────────────────────────────────────────
ttft_times      = []
latency_times   = []
decode_times    = []
inference_times = []  # prefill + decode

# ── Warm-up ONCE, outside the measurement loop ────────────────────
for _ in range(5):
    llm.generate(
        prompts=[{"prompt_token_ids": dummy_prompt_token_ids}],
        sampling_params=SamplingParams(max_tokens=50, temperature=0.0, seed=SEED),
        use_tqdm=False,
    )

# ── Measurement loop ──────────────────────────────────────────────
N_RUNS = 30
for _ in range(N_RUNS):
    output  = llm.generate(
        prompts=[{"prompt_token_ids": prompt_token_ids}],
        sampling_params=sampling_params,
        use_tqdm=False,
    )
    metrics = output[0].metrics

    ttft      = metrics.first_token_ts - metrics.scheduled_ts
    latency   = metrics.last_token_ts  - metrics.queued_ts
    decode    = metrics.last_token_ts  - metrics.first_token_ts
    inference = ttft + decode                                   # prefill + decode

    ttft_times.append(ttft)
    latency_times.append(latency)
    decode_times.append(decode)
    inference_times.append(inference)

# ── Stats ─────────────────────────────────────────────────────────
ttft_arr, latency_arr, decode_arr, inference_arr = (
    np.array(ttft_times),
    np.array(latency_times),
    np.array(decode_times),
    np.array(inference_times),
)

ttft_mean,      ttft_std      = ttft_arr.mean(),      ttft_arr.std()
latency_mean,   latency_std   = latency_arr.mean(),   latency_arr.std()
inference_mean, inference_std = inference_arr.mean(), inference_arr.std()

# decode throughput: excludes first token, over decode-only window
decode_throughput_arr                           = (MAX_GENERATION_TOKENS - 1) / decode_arr
decode_throughput_mean, decode_throughput_std   = (
    decode_throughput_arr.mean(), decode_throughput_arr.std()
)

# overall throughput: all tokens over e2e latency
overall_throughput_arr                          = MAX_GENERATION_TOKENS / latency_arr
overall_throughput_mean, overall_throughput_std = (
    overall_throughput_arr.mean(), overall_throughput_arr.std()
)

input_tokens           = len(prompt_token_ids)
total_generated_tokens = MAX_GENERATION_TOKENS

INFO 05-22 12:31:26 [loggers.py:259] Engine 000: Avg prompt throughput: 11.1 tokens/s, Avg generation throughput: 47.3 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.1%, Prefix cache hit rate: 0.0%
INFO 05-22 12:31:36 [loggers.py:259] Engine 000: Avg prompt throughput: 7.6 tokens/s, Avg generation throughput: 54.3 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.1%, Prefix cache hit rate: 0.0%
INFO 05-22 12:31:46 [loggers.py:259] Engine 000: Avg prompt throughput: 7.6 tokens/s, Avg generation throughput: 53.9 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
INFO 05-22 12:31:56 [loggers.py:259] Engine 000: Avg prompt throughput: 5.7 tokens/s, Avg generation throughput: 53.5 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.1%, Prefix cache hit rate: 0.0%
INFO 05-22 12:32:06 [loggers.py:259] Engine 000: Avg prompt throughput: 7.6 tokens/s, Avg generation throughput: 52.5 tokens/s, Running

In [13]:
print(latency_times)

[2.7447914340000352, 2.7474423669998487, 2.754153587000019, 2.75882632999992, 2.7689495470001475, 2.770416990000058, 2.7747449539999707, 2.7819026290001148, 2.786828172000014, 2.7922537590000047, 2.8010755119998976, 2.8084020430001146, 2.815388197999937, 2.8322834540001622, 2.851323088000072, 2.8699408440002117, 2.904843698999912, 2.936317754999891, 3.008541529000013, 3.0251542529999824, 3.0254003820000435, 3.1512895729999855, 3.3972081609999805, 3.5464743830000316, 3.504176153999879, 3.329351812999903, 3.111147272000153, 3.0384016579998843, 3.020711209999945, 3.020110460000069]


In [14]:
print(ttft_times)

[0.02980715999979111, 0.029649400000153037, 0.029646595000031084, 0.030851040000015928, 0.03483783799993034, 0.03231882900013261, 0.03142194100018969, 0.02966718200013929, 0.030318217000058212, 0.030397075000109908, 0.030498069999794097, 0.03342773600002147, 0.03225104700004522, 0.03215478300012364, 0.031018419000020003, 0.03185218599992368, 0.03197818299986466, 0.0337253810000675, 0.031307402000038564, 0.032258884999919246, 0.03320436400008475, 0.03231010500007869, 0.03267569300010109, 0.03409294199991564, 0.033329259000083766, 0.0329117999999653, 0.03279356600000938, 0.03246155000010731, 0.03223493999985294, 0.032520518999945125]


# Results

**Writing in json file**

In [ ]:
benchmark_results = {
    "model_name"                        : MODEL_NAME,
    "model_type"                       : MODEL_TYPE,
    "compression_method"               : COMPRESSION_METHOD,
    #"sparsity"                         : SPARSITY,
    "input_token_count"                 : input_tokens,
    "generated_token_count"             : total_generated_tokens,
    "ttft_sec"                          : f"{ttft_mean:.2f} ± {ttft_std:.2f}",
    "inference_sec"                     : f"{inference_mean:.2f} ± {inference_std:.2f}",
    "latency_sec"                       : f"{latency_mean:.2f} ± {latency_std:.2f}",
    "decode_throughput_tokens_per_sec"  : f"{decode_throughput_mean:.2f} ± {decode_throughput_std:.2f}",
    "overall_throughput_tokens_per_sec" : f"{overall_throughput_mean:.2f} ± {overall_throughput_std:.2f}",
}

# Save Results
with open(OUTPUT_FILE, "w", encoding="utf-8") as file:
    json.dump(benchmark_results, file, indent=4, ensure_ascii=False)

print(f"[INFO] Metrics successfully saved to '{OUTPUT_FILE}'")

[INFO] Metrics successfully saved to 'model_metrics.json'


**Push Results to GitHub Repository**

In [ ]:
# Paths & Repository Setup
target_dir_in_repo = f"Evaluations/InferenceEvaluations/EdgeEvaluation/{MODEL}/{MODEL_NAME_IN_REPO}/Sparsity/{SPARSITY}"
source_file = OUTPUT_FILE 
remote_url = f"https://{GIT_TOKEN}@github.com/{REPO}.git"


# Local temporary directory
current_dir = os.getcwd()
temp_repo_dir = os.path.join(current_dir, "temp_git_repo")


# Clean Previous Runs
if os.path.exists(temp_repo_dir):
    shutil.rmtree(temp_repo_dir)


# Clone Repository
print(f"Cloning repository into: {temp_repo_dir}")
subprocess.run(
    ["git", "clone", remote_url, temp_repo_dir],
    check=True
)


# Prepare Target Directory
full_target_path = os.path.join(temp_repo_dir, target_dir_in_repo)
os.makedirs(full_target_path, exist_ok=True)


# Copy Source File (FIXED)
if os.path.exists(source_file):
    dest_path = os.path.join(full_target_path, os.path.basename(source_file))
    shutil.copy2(source_file, dest_path)
else:
    print(f"Warning: Source file '{source_file}' does not exist.")


# Git Config, Commit & Push
try:
    subprocess.run(
        ["git", "-C", temp_repo_dir, "config", "user.email", EMAIL],
        check=True
    )
    subprocess.run(
        ["git", "-C", temp_repo_dir, "config", "user.name", USER_NAME],
        check=True
    )

    subprocess.run(
        ["git", "-C", temp_repo_dir, "add", "."],
        check=True
    )

    commit_msg = f"Added the cloud inference evaluation results to {target_dir_in_repo}"
    subprocess.run(
        ["git", "-C", temp_repo_dir, "commit", "-m", commit_msg],
        check=True
    )

    subprocess.run(
        ["git", "-C", temp_repo_dir, "push", "origin", "main"],
        check=True
    )

    print(f"Successfully pushed to '{REPO}' → '{target_dir_in_repo}'")

except subprocess.CalledProcessError as error:
    print(f"Git operation failed: {error}")

Cloning repository into: /kaggle/working/temp_git_repo


Cloning into '/kaggle/working/temp_git_repo'...


[main 4a86107] Added the cloud inference evaluation results to Evaluations/InferenceEvaluations/CloudEvaluation/Phi-4-mini-instruct/Phi-4-mini-instruct-Original
 1 file changed, 10 insertions(+)
 create mode 100644 Evaluations/InferenceEvaluations/CloudEvaluation/Phi-4-mini-instruct/Phi-4-mini-instruct-Original/model_metrics.json
Successfully pushed to 'MahmoudOsama20/EdgeCompress' → 'Evaluations/InferenceEvaluations/CloudEvaluation/Phi-4-mini-instruct/Phi-4-mini-instruct-Original'


To https://github.com/MahmoudOsama20/EdgeCompress.git
   46b4182..4a86107  main -> main
